# Qwen2.5-1.5B-Instruct 医疗推理 SFT（LoRA，Mac / MPS）

目标：用 `medical-o1-reasoning-SFT` 数据微调 Qwen2.5-1.5B-Instruct，让模型先推理（`<think>`）再给出诊断结论。

## 1. 环境与随机种子
导入依赖，确认 Apple MPS 可用，固定随机种子。

In [1]:
import os
import random

import numpy as np
import pandas as pd
import torch
from datasets import load_dataset, Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    DataCollatorForLanguageModeling,
    set_seed,
)
from peft import LoraConfig, TaskType, get_peft_model, PeftModel
from trl import SFTTrainer, SFTConfig

# Mac：直接使用 Apple MPS
assert torch.backends.mps.is_available(), "MPS 不可用"
device = "mps"

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
set_seed(SEED)

print(f"torch {torch.__version__} | device: {device}")

/opt/miniconda3/envs/msds692/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch 2.10.0 | device: mps


## 2. 加载数据集
读取英文子集（字段：`Question`、`Complex_CoT`、`Response`），并打印前几条样本检查内容。

In [2]:
from datasets import load_dataset

# 先流式读取英文子集，检查真实字段和样本质量
ds = load_dataset(
    "FreedomIntelligence/medical-o1-reasoning-SFT",
    "en",
    split="train"
)

for example in ds.take(3):
    print(example.keys())
    print("Question:", example["Question"])
    print("Response:", example["Response"])
    print()

dict_keys(['Question', 'Complex_CoT', 'Response'])
Question: Given the symptoms of sudden weakness in the left arm and leg, recent long-distance travel, and the presence of swollen and tender right lower leg, what specific cardiac abnormality is most likely to be found upon further evaluation that could explain these findings?
Response: The specific cardiac abnormality most likely to be found in this scenario is a patent foramen ovale (PFO). This condition could allow a blood clot from the venous system, such as one from a deep vein thrombosis in the leg, to bypass the lungs and pass directly into the arterial circulation. This can occur when the clot moves from the right atrium to the left atrium through the PFO. Once in the arterial system, the clot can travel to the brain, potentially causing an embolic stroke, which would explain the sudden weakness in the left arm and leg. The connection between the recent travel, which increases the risk of deep vein thrombosis, and the neurologi

## 3. 划分训练集 / 测试集
按 9:1 随机划分，得到 `split["train"]` 和 `split["test"]`。

In [3]:
split = ds.train_test_split(test_size=0.1, seed=SEED)
split

DatasetDict({
    train: Dataset({
        features: ['Question', 'Complex_CoT', 'Response'],
        num_rows: 17733
    })
    test: Dataset({
        features: ['Question', 'Complex_CoT', 'Response'],
        num_rows: 1971
    })
})

## 4. 数据构建
- 转成对话格式：`prompt` = 问题，`completion` = `<think>推理过程</think>` + 最终回答
- 用 Qwen chat template 统计 token 数，丢弃超过 `MAX_LEN=1024` 的样本（不截断，避免切掉答案）
- 先取小规模子集跑通（`N_TRAIN`、`N_EVAL`），`N_TRAIN=None` 则用全量训练集

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
MAX_LEN = 1024
N_TRAIN, N_EVAL = 800, 50  # 先小规模跑通；N_TRAIN 设为 None 则用全量训练集

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def to_chat(ex):
    # 推理过程放进 <think>，最终结论放在其后，让模型学会先推理再作答
    return {
        "prompt": [{"role": "user", "content": ex["Question"]}],
        "completion": [
            {
                "role": "assistant",
                "content": f"<think>\n{ex['Complex_CoT']}\n</think>\n\n{ex['Response']}",
            }
        ],
    }


def n_tokens(ex):
    text = tokenizer.apply_chat_template(ex["prompt"] + ex["completion"], tokenize=False)
    return {"n_tokens": len(tokenizer(text).input_ids)}


def prepare(d):
    d = d.map(to_chat, remove_columns=d.column_names).map(n_tokens)
    # 超长样本直接丢弃：截断会把结尾的最终答案切掉
    return d.filter(lambda ex: ex["n_tokens"] <= MAX_LEN).remove_columns("n_tokens")


# split 已打乱，直接取前 N 条即可
train_raw = split["train"].select(range(N_TRAIN)) if N_TRAIN else split["train"]
train_ds = prepare(train_raw)
eval_ds = prepare(split["test"].select(range(N_EVAL)))
print(f"train: {len(train_raw)} -> {len(train_ds)} | eval: {N_EVAL} -> {len(eval_ds)}  (过滤超过 {MAX_LEN} tokens 的样本)")

print(tokenizer.apply_chat_template(
    train_ds[0]["prompt"] + train_ds[0]["completion"], tokenize=False
)[:800])

## 5. 加载基础模型 + 微调前基线
以 bf16 把 Qwen2.5-1.5B-Instruct 加载到 MPS，定义生成函数 `ask`，先记录微调前在验证集前几题上的回答，用于之后对比。

In [ ]:
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.bfloat16).to(device)

GEN_TOKENS = 768
N_COMPARE = 2  # 用验证集前几题对比微调前后


def ask(m, question, max_new_tokens=GEN_TOKENS):
    inputs = tokenizer.apply_chat_template(
        [{"role": "user", "content": question}],
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    ).to(device)
    m.eval()
    with torch.no_grad():
        out = m.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, use_cache=True)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)


# 微调前基线
baseline = [ask(model, eval_ds[i]["prompt"][0]["content"]) for i in range(N_COMPARE)]
for i, ans in enumerate(baseline):
    print(f"--- baseline {i} ---\n{ans}\n")

## 6. LoRA 配置与训练
对全部 7 类线性层挂 LoRA（r=16），用 `SFTTrainer` 训练。loss 只计算回答部分。`MAX_STEPS` 控制训练步数（冒烟测试，约 1 小时 / 60 步）。

In [ ]:
OUTPUT_DIR = "outputs/qwen2.5-1.5b-medical-lora"
MAX_STEPS = 60  # 冒烟测试；确认 loss 下降后再加大训练量

peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

args = SFTConfig(
    output_dir=OUTPUT_DIR,
    max_length=MAX_LEN,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=5,
    max_steps=MAX_STEPS,
    logging_steps=5,
    eval_strategy="steps",
    eval_steps=20,
    save_strategy="steps",
    save_steps=20,
    save_total_limit=2,
    bf16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    report_to="none",
    seed=SEED,
)

trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    processing_class=tokenizer,
    peft_config=peft_config,
)
trainer.model.print_trainable_parameters()
trainer.train()

## 7. 保存 adapter 并对比微调前后
只保存 LoRA adapter，然后对同样的问题打印参考答案、基线回答和微调后回答。

In [ ]:
# 只保存 LoRA adapter（几十 MB），推理时再挂到基础模型上
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

# 微调后与基线对比
for i in range(N_COMPARE):
    ex = eval_ds[i]
    q = ex["prompt"][0]["content"]
    print(f"=== Q{i}: {q}\n")
    print(f"--- reference ---\n{ex['completion'][0]['content']}\n")
    print(f"--- baseline ---\n{baseline[i]}\n")
    print(f"--- fine-tuned ---\n{ask(trainer.model, q)}\n")

## 8. Reward 函数设计（为后续 GRPO 强化学习准备）
代码在 [medical_rewards.py](medical_rewards.py)，签名兼容 TRL `GRPOTrainer`（`reward_func(completions, **kwargs)`，数据集需带 `question`、`reference` 两列）。

| reward | 范围 | 含义 |
|---|---|---|
| `format_reward` | 0–1 | `<think>…</think>` 结构正确 (0.4) + 推理不少于 20 词 (0.3) + 答案非空且无残留标签 (0.3) |
| `accuracy_reward` | 0–1 | **选择题**：还原出正确选项后做精确匹配（可验证）；**自由文本**：回答里“题目之外新引入的术语”与参考答案的 F1 |
| `repetition_penalty` | −1–0 | 4-gram 重复（循环生成）和过长的 `<think>` 扣分 |

权重 `REWARD_WEIGHTS = [0.2, 0.7, 0.1]`。准确性只看 `</think>` 之后的答案，推理部分不给分。

**已知局限**：自由文本部分是词汇重叠，不是语义判断。同义改写会被低估，同一主题下措辞相近的错误诊断可能被高估。如果之后要更可靠的信号，需要换成 LLM 打分。

In [ ]:
import importlib
import medical_rewards as R
importlib.reload(R)

think = ("Let me work through this. The presentation points to a specific mechanism, so I compare the key findings "
         "with the most likely causes, rule out the alternatives one by one, and then settle on the best explanation.")


def wrap(answer):
    return f"<think>\n{think}\n</think>\n\n{answer}"


# 用验证集里第 0 题演示：参考答案 / 错误答案 / 只抄题目 / 格式错误
ex = split["test"][0]
q, ref = ex["Question"], ex["Response"]
other = split["test"][1]["Response"]

demo = {
    "参考答案": wrap(ref),
    "别的题的答案": wrap(other),
    "只抄题目": wrap(q),
    "没有 <think>": ref,
    "被截断（没有 </think>）": f"<think>\n{think}",
}
for name, c in demo.items():
    f = R.format_reward([c])[0]
    a = R.accuracy_reward([c], question=[q], reference=[ref])[0]
    r = R.repetition_penalty([c])[0]
    t = R.total_reward([c], question=[q], reference=[ref])[0]
    print(f"{name:20s} format={f:.2f} accuracy={a:.2f} repeat={r:+.2f} | total={t:.3f}")

# 选择题：是否能还原出正确选项
mcq = [e for e in split["test"] if R.gold_option(e["Question"], e["Response"])]
print(f"\n测试集中可还原正确选项的选择题: {len(mcq)}/{len(split['test'])}")